In [7]:
import re
import numpy as np
import time

start_time = time.perf_counter()

articles = {}

# Loading the data
with open(r"C:\Users\user\Desktop\articles.txt", "r", encoding="utf-8") as f:
    for line in f:
        article_id, text = line.rstrip("\n").split(" ", 1)
        articles[article_id] = text

# Preprocessing
articles = {
    ID: re.sub(r"[^a-zA-Z\s]", "", text)
    for ID, text in articles.items()
}

articles = {ID: text.lower() for ID, text in articles.items()}

# Generate k-shingles
shingles_per_doc = {}

for ID in articles.keys():
    text = articles[ID]
    shingles = set()
    for i in range(len(text)-2):
        shingles.add(text[i:i+3])
    shingles_per_doc[ID] = shingles

pairs = []
for ID, shingles in shingles_per_doc.items():
    for shingle in shingles:
        pairs.append((shingle,ID))

pairs.sort()

all_shingles = set()

for shingles in shingles_per_doc.values():
    all_shingles.update(shingles)

# Construct the characteristic matrix T
n = len(all_shingles) # = 8747
c = 8753 
b = 6
r = 20
sig = b*r # number of signatures

shingle_indexes = { shingle: i for i, shingle in enumerate(all_shingles)}

doc_indexes = { ID: i for i, ID in enumerate(articles.keys())}

T = np.zeros((n,2000), dtype = int)
for shingle, ID in pairs:
    i = shingle_indexes[shingle]
    j = doc_indexes[ID]
    T[i][j]=1

# Generating Hash Functions
A = []
B = []
for i in range(sig):
    r1 = np.random.randint(0,c)
    A.append(r1)
    r2 = np.random.randint(0,c)
    B.append(r2)

# Computing Signatures
M = np.full((sig, 2000), np.inf)

for i in range(n):
    for j in range(2000):
        if T[i][j]==1:
            for k in range(sig):
                hk_i = (A[k]*i+B[k])%c
                if hk_i < M[k][j]:
                    M[k][j] = hk_i


# Createing candidates

candidate = set()

# Note that we use the same hash function for all bands.
# Also, the hash function for bands were not specified in the problem.
def hf(values):
    h = 0
    for x in values:
        h = (h * 103 + int(x)) % 1999
    return h
    
for i in range(b):
    buckets = {}

    for j in range(2000):
        col = tuple(M[r*i:r*i+r, j])
        bucket = hf(col)

        if bucket not in buckets:
            buckets[bucket] = []
        buckets[bucket].append(j)

    for docs in buckets.values():
        for x in range(len(docs)):
            for y in range(x + 1, len(docs)):
                candidate.add((docs[x], docs[y]))


# Now we calculate the similarity of signatures of candidate pairs
similar_pairs = []

for i, j in candidate:
    cand1 = M[:, i]
    cand2 = M[:, j]

    # Checking the similarity of the signatures
    x = 0
    for t in range(sig):
        if cand1[t] == cand2[t]:
            x += 1
            
    sim = x / sig #if (x + y) > 0 else 0

    if sim >= 0.9:
        similar_pairs.append((i, j, sim))

reverse_doc_indexes = {
    i: ID
    for i, ID in enumerate(articles.keys())
}

similar_pairs = [(reverse_doc_indexes[x[0]], reverse_doc_indexes[x[1]], x[2]) for x in similar_pairs]

for item in similar_pairs:
    print(item[0], "\t", item[1], "\t", item[2], sep="")
    
end_time = time.perf_counter()
print("Elapsed time:", end_time - start_time, "seconds")

t980	t2023	0.9833333333333333
t1621	t7958	1.0
t3268	t7998	0.9916666666666667
t448	t8535	1.0
t8413	t269	1.0
Elapsed time: 310.23365410000406 seconds


In [8]:
!jupyter nbconvert --to script hw1_3_b.ipynb

[NbConvertApp] Converting notebook hw1_3_b.ipynb to script
[NbConvertApp] Writing 3244 bytes to hw1_3_b.py
